In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

# 1. SETUP MEMORIA: DEVE ESSERE LA PRIMA COSA IN ASSOLUTO
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

# 2. ESORCISMO DELLA RAM (Uccide i vecchi modelli in memoria)
tf.keras.backend.clear_session()

# ABILITA IL GPU MEMORY GROWTH (Evita il crash VRAM)
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for gpu in physical_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth abilitato con successo.")
    except RuntimeError as e:
        print(f"Errore configurazione GPU: {e}")

I0000 00:00:1783441056.448822   45121 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.
✅ GPU Memory Growth abilitato con successo.


In [2]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    # 1. Calcolo FLASH (4 byte per Float32, 1 byte per INT8)
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    # 2. Calcolo SRAM (Tensor Arena) con logica Adiacente (Buffer Reuse)
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    # Memoria occupata dal layer precedente (inizializzata con la dimensione dell'input)
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            # Per i layer come "Concatenate" che potrebbero avere output multipli/strani
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        
        # IL FIX È QUI: Sommiamo il layer precedente e il layer corrente!
        # È il momento esatto in cui TFLM consuma più RAM durante l'esecuzione di questo layer.
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    print("============================================")
    mode_str = "INT8 (Quantizzato)" if is_int8 else "FLOAT32 (Training)"
    print(f"   REPORT REQUISITI ESP32-S3 [{mode_str}]   ")
    print("============================================")
    print(f" Memoria FLASH stimata : ~{estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{max_adjacent_ram_kb:.2f} KB (Limite: 300 KB)")
    print("============================================\n")

In [3]:
# =====================================================================
# BULGARIAN SQUAT PER MAC
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)
ROOM_DIMS = tf.constant([4.8, 7.2], dtype=tf.float32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1)) 
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1)) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

I0000 00:00:1783441060.174919   45121 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2603 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [4]:
# ==============================================================================
# DATA ENGINE (Caricamento Globale in RAM + normalizzazione [0, 1])
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.02):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy'].astype(np.float32) # Assicuriamoci sia float
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y


# ==============================================================================
# 2. SPLIT DIVERSI SEGUENDO DIVERSI CRITERI
# ==============================================================================
# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 (80% train e 22% val), stesso cocetto di split 1 "STRESS TEST sul MULTIPATH"
#val_indices = [23, 20, 0, 13, 9] 
#train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 (78% train e 22% val) > equilibrato tra train e val 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

tutti_i_file = glob.glob("dataset/data/*.npz")
#tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 2. CARICAMENTO DATI GREZZI
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

# ==============================================================================
# 3. NORMALIZZAZIONE X (INPUT) CON GLOBAL MAX
# ==============================================================================
GLOBAL_MAX = np.percentile(X_train_raw, 99.5)
print(f"\n---> GLOBAL_MAX CALCOLATO: {GLOBAL_MAX:.2f} <---")
print("INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!")

X_train = np.clip(X_train_raw, 0, GLOBAL_MAX) / GLOBAL_MAX
X_val = np.clip(X_val_raw, 0, GLOBAL_MAX) / GLOBAL_MAX

print("\n==================================================")
print(f"DATI TOTALI PRONTI E NORMALIZZATI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

---> GLOBAL_MAX CALCOLATO: 203.93 <---
INSERISCI QUESTO VALORE NEL TUO CODE.PY PER L'INFERENZA!

DATI TOTALI PRONTI E NORMALIZZATI IN RAM!
Totale FRAME individuali di Train:      135000
Totale FRAME individuali di Validation: 45000


In [5]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 - RESIDUAL REDUCTION MODULES (RRM)
# ==============================================================================

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze: estrae le statistiche globali per ogni canale
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation: riduce e poi ri-espande per imparare i pesi ottimali
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape per applicare il broadcasting moltiplicativo
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d(x, filters, r=8, name_prefix=""):
    """
    RRM: Red(Res(x)) 
    Unisce una skip connection pesata dal SE block e un dimezzamento 
    parallelo della dimensione temporale (range bins).
    """
    # --- 1. Residual Branch (Res) ---
    res = layers.Conv2D(filters, kernel_size=(1, 3), padding='same', activation='relu', 
                        name=f"{name_prefix}_res_conv")(x)
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x]) # Skip connection

    # --- 2. Reduction Branch (Red) ---
    # Due convoluzioni parallele con stride=(1,2) per dimezzare i range bins (da 120->60->30...)
    red1 = layers.Conv2D(filters, kernel_size=(1, 3), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv1")(res)
    # Kernel 1x1 funge da projection mapping tipico delle ResNet
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 # Numero di filtri base: mantiene il modello piccolo e potente
    r = 8  # Reduction ratio per il blocco SE (come da paper)
    
    # 1. Feature Extraction Iniziale (allinea il numero di canali a F per far funzionare le Add)
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(inputs)
    
    # 2. Cascata di Residual Reduction Modules
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    # Excitation: riduce e poi ri-espande per imparare
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    # 3. Testa della rete (Flatten + Dense)
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.25, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # 4. Multi-Head Output (Coordinate + Maschera presenze)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [5]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 - RESIDUAL REDUCTION MODULES (RRM) CON DEPTHWISE SEPARABLE CONV
# ==============================================================================

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze: estrae le statistiche globali per ogni canale
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation: riduce e poi ri-espande per imparare i pesi ottimali
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape per applicare il broadcasting moltiplicativo
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d_mobile(x, filters, r=8, name_prefix=""):
    # --- 1. Residual Branch (Res) ---
    # Depthwise Separable invece di Conv2D standard
    res = layers.DepthwiseConv2D(kernel_size=(1, 3), padding='same', use_bias=False, name=f"{name_prefix}_res_dw")(x)
    res = layers.BatchNormalization(name=f"{name_prefix}_res_bn1")(res)
    res = layers.ReLU(name=f"{name_prefix}_res_relu1")(res)
    res = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_res_pw")(res)
    
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x])

    # --- 2. Reduction Branch (Red) ---
    red1 = layers.DepthwiseConv2D(kernel_size=(1, 3), strides=(1, 2), padding='same', use_bias=False, name=f"{name_prefix}_red_dw")(res)
    red1 = layers.BatchNormalization(name=f"{name_prefix}_red_bn2")(red1)
    red1 = layers.ReLU(name=f"{name_prefix}_red_relu2")(red1)
    red1 = layers.Conv2D(filters, kernel_size=(1, 1), padding='same', activation='relu', name=f"{name_prefix}_red_pw")(red1)
    
    # red2 resta una Conv2D standard 1x1 (è già il metodo più economico)
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 64 # Numero di filtri base: mantiene il modello piccolo e potente
    r = 8  # Reduction ratio per il blocco SE (come da paper)
    
    # 1. Feature Extraction Iniziale (allinea il numero di canali a F per far funzionare le Add)
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(inputs)
    
    # 2. Cascata di Residual Reduction Modules
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    # Excitation: riduce e poi ri-espande per imparare
    x = residual_reduction_module_2d_mobile(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    # 3. Testa della rete (Flatten + Dense)
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.35, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # 4. Multi-Head Output (Coordinate + Maschera presenze)
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [6]:
print(np.max(X_train))
print(np.mean(X_train))

1.0
0.060004964


In [6]:
# ==============================================================================
# ADDESTRAMENTO MODELLO V2 (RRM)
# ==============================================================================

print("\n--- PREPARAZIONE TF.DATA PIPELINE ---")
# CREIAMO I DATASET (Previene la creazione di un singolo tensore gigante)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
# Shuffle, batch e prefetch massimizzano l'uso della GPU senza saturarla
train_dataset = train_dataset.shuffle(buffer_size=5000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, Y_val))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

# Inizializzazione del nuovo modello
model_rrm = build_eeai_model_v2_rrm()

# Compilazione 
model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("norm+depth+drop_model_toscano.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50
embedded_summary(model_rrm, input_shape=(1, 120, 18), is_int8=False)

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM ---")
history_rrm = model_rrm.fit(
    train_dataset,
    validation_data= val_dataset,
    # batch_size=32,
    # shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- PREPARAZIONE TF.DATA PIPELINE ---


W0000 00:00:1783441085.521257   45121 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.
W0000 00:00:1783441086.743323   45121 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.


   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~510.30 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)


--- INIZIO ADDESTRAMENTO MODELLO RRM ---
Epoch 1/50


W0000 00:00:1783441088.772598   45121 cpu_allocator_impl.cc:82] Allocation of 1166400000 exceeds 10% of free system memory.
I0000 00:00:1783441093.829128   45187 service.cc:153] XLA service 0x7fc3f4052e30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783441093.829147   45187 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1650, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.10.1)
I0000 00:00:1783441093.991641   45187 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783441095.113989   45187 cuda_dnn.cc:461] Loaded cuDNN version 91001
I0000 00:00:1783441095.244267   45187 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13000__.103
I0000 00:00:1783441108.091848   45187 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4210/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9600 - hungarian_rmse_metres: 0.7071 - loss: 0.8999

I0000 00:00:1783441135.008710   45187 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_13000__.103


4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - hungarian_mask_acc: 0.9601 - hungarian_rmse_metres: 0.7067 - loss: 0.8992

I0000 00:00:1783441150.304902   45187 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_35285__.20
I0000 00:00:1783441154.734898   45187 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_35285__.20



Epoch 1: val_loss improved from None to 1.36960, saving model to norm+depth+drop_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 71s 12ms/step - hungarian_mask_acc: 0.9762 - hungarian_rmse_metres: 0.5435 - loss: 0.5629 - val_hungarian_mask_acc: 0.7961 - val_hungarian_rmse_metres: 0.5417 - val_loss: 1.3696 - learning_rate: 0.0010
Epoch 2/50
4211/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9915 - hungarian_rmse_metres: 0.4916 - loss: 0.3781
Epoch 2: val_loss improved from 1.36960 to 1.33290, saving model to norm+depth+drop_model_toscano.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 27s 6ms/step - hungarian_mask_acc: 0.9884 - hungarian_rmse_metres: 0.4423 - loss: 0.3525 - val_hungarian_mask_acc: 0.8117 - val_hungarian_rmse_metres: 0.5344 - val_loss: 1.3329 - learning_rate: 0.0010
Epoch 3/50
4217/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - hungarian_mask_acc: 0.9929 - hungarian_rmse_metres: 0.4417 - loss: 0.3127
Epoch 3: val_loss improved from 1.33290 to 1.12255, saving model to n

NORM: val_hungarian_mask_acc: 0.9487 - val_hungarian_rmse_metres: 0.4439
NORM + DEPTH: val_hungarian_mask_acc: 0.9411 - val_hungarian_rmse_metres: 0.4487
NORM + DEPTH + 0.35 DROPOUT: val_hungarian_mask_acc: 0.9479 - val_hungarian_rmse_metres: 0.4680
NORM + DEPTH + f=128 :val_hungarian_mask_acc: 0.9361 - val_hungarian_rmse_metres: 0.4729 